SOAL 1 — Data Loading & Profiling Awal

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StructType, StructField, StringType, DateType, TimestampType

In [2]:
spark = SparkSession.builder \
    .appName("ecommerce_profiling") \
    .master("local[*]") \
    .getOrCreate()

BASE = "dataset/"


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/giwang.kintan/Documents/DE/pyspark_assignment/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/17 19:03:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/17 19:03:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/17 19:03:14 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
spark.sparkContext.setLogLevel("ERROR")

In [4]:
orders_schema = StructType([
    StructField("order_id",     StringType()),
    StructField("user_id",      StringType()),
    StructField("order_date",   DateType()),
    StructField("total_amount", DoubleType()),
])
order_items_schema = StructType([
    StructField("order_id",      StringType()),
    StructField("product_id",    StringType()),
    StructField("quantity",      IntegerType()),
    StructField("price",         DoubleType()),
])
products_schema = StructType([
    StructField("product_id",   StringType()),
    StructField("product_name", StringType()),
    StructField("category",     StringType()),
    StructField("price",        DoubleType()),
])
users_schema = StructType([
    StructField("user_id",     StringType()),
    StructField("name",        StringType()),
    StructField("city",        StringType()),
    StructField("signup_date", DateType()),
])

In [5]:
def load(name, schema):
    return (spark.read.option("header", True)
            .option("dateFormat", "yyyy-MM-dd")
            .option("mode", "PERMISSIVE")
            .schema(schema)
            .csv(f"{BASE}{name}.csv"))

In [6]:

orders      = load("orders", orders_schema)
order_items = load("order_items", order_items_schema)
products    = load("products", products_schema)
users       = load("users", users_schema)

dfs = {"orders": orders, "order_items": order_items,
       "products": products, "users": users}

In [7]:
print("=== Jumlah baris tiap DataFrame ===")
for table, row in dfs.items():
    print(f"{table}: {row.count()}")


print("\n=== cek null per kolom ===")
for table, row in dfs.items():
    print(f"\n[{table}]")
    row.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                for c in row.columns]).show(truncate=False)

print("\n=== Sample 5 baris ===")
for table, row in dfs.items():
    print(f"\n[{table}]")
    row.show(5, truncate=False)


=== Jumlah baris tiap DataFrame ===
orders: 18
order_items: 20
products: 10
users: 14

=== cek null per kolom ===

[orders]
+--------+-------+----------+------------+
|order_id|user_id|order_date|total_amount|
+--------+-------+----------+------------+
|1       |1      |0         |0           |
+--------+-------+----------+------------+


[order_items]
+--------+----------+--------+-----+
|order_id|product_id|quantity|price|
+--------+----------+--------+-----+
|1       |1         |0       |0    |
+--------+----------+--------+-----+


[products]
+----------+------------+--------+-----+
|product_id|product_name|category|price|
+----------+------------+--------+-----+
|1         |0           |0       |0    |
+----------+------------+--------+-----+


[users]
+-------+----+----+-----------+
|user_id|name|city|signup_date|
+-------+----+----+-----------+
|1      |0   |0   |0          |
+-------+----+----+-----------+


=== Sample 5 baris ===

[orders]
+--------+-------+----------+--------

In [8]:
print("=== jumlah orders dengan total_amount <= 0 ===")

orders_bad_amount = orders.filter(F.col("total_amount") <= 0).count()
print(f"orders: total_amount <= 0: {orders_bad_amount}")

print("\n=== jumlah order_items dengan quantity <= 0 atau price <= 0 ===")
items_bad_qty_or_price = order_items.filter(
    (F.col("quantity") <= 0) | (F.col("price") <= 0)
).count()

print(f"order_items: quantity <= 0 or price <= 0: {items_bad_qty_or_price}")

print("\n=== jumlah duplikasi key utama (order_id di orders, product_id di products, user_id di users) ===")
def count_duplicates(df, key):
    return df.groupBy(key).count().filter(F.col("count") > 1).count()

print(f"orders: duplicate order_id: {count_duplicates(orders, 'order_id')}")
print(f"products: duplicate product_id: {count_duplicates(products, 'product_id')}")
print(f"users: duplicate user_id: {count_duplicates(users, 'user_id')}")




=== jumlah orders dengan total_amount <= 0 ===
orders: total_amount <= 0: 2

=== jumlah order_items dengan quantity <= 0 atau price <= 0 ===
order_items: quantity <= 0 or price <= 0: 2

=== jumlah duplikasi key utama (order_id di orders, product_id di products, user_id di users) ===
orders: duplicate order_id: 1
products: duplicate product_id: 1
users: duplicate user_id: 1


SOAL 2 — Cleaning & Standardization
Perintah 1:
Bersihkan data:
drop baris orders yang order_id atau user_id null,
drop baris order_items yang order_id atau product_id null,
drop baris products yang product_id null,
drop baris users yang user_id null.

Perintah 2:
Standardisasi kolom teks:
city di users → lowercase + trim,
category di products → lowercase + trim.
Perintah 3:
Deduplication berbasis key:
orders dedupe by order_id,
order_items dedupe by kombinasi (order_id,product_id),
products dedupe by product_id,
users dedupe by user_id.
 Tunjukkan jumlah baris sebelum vs sesudah dedupe.


In [9]:
# ================ Perintah 1: Bersihkan data ======================

print("=== drop baris yang null===")
orders_cleaned = orders.dropna(subset=["order_id", "user_id"])
order_items_cleaned = order_items.dropna(subset=[ "order_id", "product_id"])
products_cleaned = products.dropna(subset=["product_id"])
users_cleaned = users.dropna(subset=["user_id"])

print("drop baris yang null selesai")


=== drop baris yang null===
drop baris yang null selesai


In [10]:

# ================ Perintah 2: Standardisasi kolom teks ======================
users_cleaned = users_cleaned.withColumn("city", F.lower(F.trim(F.col("city"))))
products_cleaned = products_cleaned.withColumn("category", F.lower(F.trim(F.col("category"))))

users_cleaned.show(5, truncate=False)
products_cleaned.show(5, truncate=False)


+-------+-------------+--------+-----------+
|user_id|name         |city    |signup_date|
+-------+-------------+--------+-----------+
|U001   |Andi Pratama |jakarta |2023-01-15 |
|U002   |Budi Santoso |surabaya|2023-03-20 |
|U002   |Budi Santoso |surabaya|2023-03-20 |
|U003   |Citra Dewi   |bandung |2023-06-10 |
|U004   |Dimas Anggara|jakarta |2023-07-01 |
+-------+-------------+--------+-----------+
only showing top 5 rows
+----------+-------------+----------+---------+
|product_id|product_name |category  |price    |
+----------+-------------+----------+---------+
|P001      |Smartphone X |elektronik|3500000.0|
|P001      |Smartphone X |elektronik|3500000.0|
|P002      |Laptop Pro   |elektronik|8500000.0|
|P003      |Kemeja Casual|fashion   |150000.0 |
|P004      |Celana Jeans |fashion   |250000.0 |
+----------+-------------+----------+---------+
only showing top 5 rows


In [11]:
# ================ Perintah 3: Deduplication berbasis key: ======================
orders_dedup = orders_cleaned.dropDuplicates(["order_id"])
order_items_dedup = order_items_cleaned.dropDuplicates(["order_id", "product_id"])
products_dedup = products_cleaned.dropDuplicates(["product_id"])
users_dedup = users_cleaned.dropDuplicates(["user_id"])

# ================ Perintah 4: Data sebelum vs sesudah dedupe ======================
pairs = [
    ("orders",      orders_cleaned,      orders_dedup),
    ("order_items", order_items_cleaned, order_items_dedup),
    ("products",    products_cleaned,    products_dedup),
    ("users",       users_cleaned,       users_dedup),
]
rows = []
for name, before_df, after_df in pairs:
    before, after = before_df.count(), after_df.count()
    rows.append((name, before, after, before - after))

print("\n=== jumlah baris sebelum vs sesudah deduplication ===")
dedup_summary = spark.createDataFrame(
    rows, ["table", "before_dedupe", "after_dedupe", "rows_removed"]
)
dedup_summary.show(truncate=False)



=== jumlah baris sebelum vs sesudah deduplication ===


26/10/17 19:03:36 ERROR Executor: Exception in task 0.0 in stage 90.0 (TID 55)
org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "/Users/giwang.kintan/Documents/DE/pyspark_assignment/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3569, in main
    check_python_version(infile)
  File "/Users/giwang.kintan/Documents/DE/pyspark_assignment/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker_util.py", line 86, in check_python_version
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version: 3.11 than that in driver: 3.12, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.
 SQLSTATE: 38000
	at org.apache.spark.api.python.BasePythonRunne

PythonException: An exception was thrown from the Python worker:
Traceback (most recent call last):
  File "/Users/giwang.kintan/Documents/DE/pyspark_assignment/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3569, in main
    check_python_version(infile)
  File "/Users/giwang.kintan/Documents/DE/pyspark_assignment/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker_util.py", line 86, in check_python_version
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version: 3.11 than that in driver: 3.12, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

SOAL 3 — Data Enrichment & Feature Engineering
Perintah 1:
Di order_items, buat kolom turunan:
total_item_value = quantity * price.
Perintah 2:
Di orders, buat kolom kategorisasi menggunakan when/otherwise:
order_value_flag dengan aturan:
high_value jika total_amount > 500000, selain itu normal.
Perintah 3:
Buat kolom segmentasi user di users:
user_tenure_days = datediff(order_date_max_per_user, signup_date) (Hint: butuh join/agg kecil untuk order_date_max_per_user).
tenure_segment = new (<=30 hari), mature (31–180), loyal (>180).

In [ ]:
# ================ Perintah 1: Di order_items, buat kolom turunan ======================
order_items_cleaned = order_items_cleaned.withColumn("total_price", F.col("quantity") * F.col("price"))
# ================ Perintah 2: Di orders, buat kolom kategorisasi menggunakan when/otherwise:======================
orders_cleaned = orders_cleaned.withColumn("order_value_flag", F.when(F.col("total_amount") > 500000, "high_value").otherwise("normal"))    
# ================ Perintah 3: Di users, buat kolom segmentasi ======================
users_cleaned = users_cleaned.withColumn("user_tenure_days", F.datediff(F.current_date(), F.col("signup_date")))
users_cleaned = users_cleaned.withColumn("tenure_segment", F.when(F.col("user_tenure_days") > 365, "loyal").otherwise("new"))